In [ ]:
# ----------------------------CONFIGURACIÓN INICIAL----------------------------


# Fijar semilla para reproducibilidad
semilla = 12345  # CAMBIAR POR VUESTRO NIA
import numpy as np
np.random.seed(semilla)

# Importación de librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

# Preprocesamiento y pipelines
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Métricas
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report, make_scorer

# Configuración de visualización
plt.style.use('ggplot')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(" Configuración inicial completada. Semilla fijada:", semilla)


# ----------------------------CARGA DE DATOS----------------------------

# Cargar el archivo específico del grupo (bank_XX.pkl)
# SUSTITUIR 'XX' por los dos últimos dígitos del NIA elegido

df = pd.read_pickle('bank_XX.pkl')  # Ejemplo: bank_45.pkl

print(" Datos cargados correctamente")
print(f"Dimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
print("\nPrimeras 5 filas:")
df.head()

# Información general del dataset
df.info()


# --------------------------1. EDA SIMPLIFICADO----------------------------

# 1.1 IDENTIFICACIÓN DE VARIABLES


# Separar variable objetivo
target = 'deposit'
y = df[target]
X = df.drop(columns=[target])

# Identificar tipos de variables automáticamente
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("=" * 50)
print("IDENTIFICACIÓN DE VARIABLES")
print("=" * 50)
print(f"Variables numéricas ({len(num_cols)}): {num_cols}")
print(f"\nVariables categóricas ({len(cat_cols)}): {cat_cols}")

# 1.2 CARDINALIDAD EN VARIABLES CATEGÓRICAS


print("=" * 50)
print("ANÁLISIS DE CARDINALIDAD")
print("=" * 50)

alta_cardinalidad = []
for col in cat_cols:
    n_unique = X[col].nunique()
    print(f"{col}: {n_unique} valores únicos")
    if n_unique > 10:
        alta_cardinalidad.append(col)
        print(f"  ¡ALTA CARDINALIDAD! (>10 valores)")

print(f"\n Variables con alta cardinalidad: {alta_cardinalidad}")


# 1.3 VALORES FALTANTES


print("=" * 50)
print("ANÁLISIS DE VALORES FALTANTES")
print("=" * 50)

missing_total = X.isnull().sum()
missing_percent = (missing_total / len(X)) * 100

missing_df = pd.DataFrame({
    'Variable': missing_total.index,
    'Faltantes': missing_total.values,
    'Porcentaje': missing_percent.values
})
missing_df = missing_df[missing_df['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

if len(missing_df) > 0:
    print("Variables con valores faltantes:")
    print(missing_df.to_string(index=False))
    
    # Visualización
    plt.figure(figsize=(10, 5))
    sns.barplot(data=missing_df, x='Variable', y='Porcentaje')
    plt.title('Porcentaje de valores faltantes por variable')
    plt.xticks(rotation=45)
    plt.ylabel('Porcentaje (%)')
    plt.tight_layout()
    plt.show()
else:
    print("No hay valores faltantes en el dataset")


# 1.4 COLUMNAS CONSTANTES O POSIBLES IDS

print("=" * 50)
print("ANÁLISIS DE COLUMNAS CONSTANTES O ID")
print("=" * 50)

# Columnas constantes (un solo valor)
constantes = []
for col in X.columns:
    if X[col].nunique() == 1:
        constantes.append(col)
        print(f"Columna constante: {col} (valor único: {X[col].iloc[0]})")

# Posibles columnas ID (valores únicos = número de filas)
posibles_ids = []
for col in X.columns:
    if X[col].nunique() == len(X):
        posibles_ids.append(col)
        print(f"Posible columna ID: {col} (todos los valores únicos)")

if not constantes and not posibles_ids:
    print("No se detectaron columnas constantes ni posibles IDs")


# 1.5 ANÁLISIS ESPECIAL: pdays


print("=" * 50)
print("ANÁLISIS ESPECIAL DE LA VARIABLE pdays")
print("=" * 50)

# Análisis descriptivo
print("Estadísticas descriptivas de pdays:")
print(X['pdays'].describe())

# Análisis del valor -1 (sin contacto previo)
n_minus_one = (X['pdays'] == -1).sum()
pct_minus_one = (n_minus_one / len(X)) * 100
print(f"\nValor -1 (sin contacto previo): {n_minus_one} instancias ({pct_minus_one:.2f}%)")

# Distribución de pdays (excluyendo -1)
pdays_sin_menos1 = X[X['pdays'] != -1]['pdays']
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(pdays_sin_menos1, bins=30, kde=True)
plt.title('Distribución de pdays (excluyendo -1)')
plt.xlabel('pdays')

plt.subplot(1, 2, 2)
sns.boxplot(x=pdays_sin_menos1)
plt.title('Boxplot de pdays (excluyendo -1)')
plt.xlabel('pdays')

plt.tight_layout()
plt.show()


# ----------------------------PREPROCESO DE pdays----------------------------


print("\n APLICANDO PREPROCESO DE pdays:")

# Crear copia para el preproceso
X_preprocesado = X.copy()

# Opción elegida: 
# 1. Crear variable binaria 'was_contacted_before'
# 2. Convertir -1 a NaN para luego imputar

X_preprocesado['was_contacted_before'] = (X_preprocesado['pdays'] != -1).astype(int)
X_preprocesado.loc[X_preprocesado['pdays'] == -1, 'pdays'] = np.nan

print("Creada nueva variable: 'was_contacted_before'")
print("Valores -1 convertidos a NaN para su posterior imputación")

# Verificar el resultado
print("\nPrimeras 10 filas después del preproceso:")
print(X_preprocesado[['pdays', 'was_contacted_before']].head(10))


# 1.6 ANÁLISIS DE LA VARIABLE OBJETIVO


print("=" * 50)
print("ANÁLISIS DE LA VARIABLE OBJETIVO")
print("=" * 50)

# Distribución de clases
print("Distribución de 'deposit':")
print(y.value_counts())
print(f"\nPorcentajes:")
print(y.value_counts(normalize=True) * 100)

# Visualización
plt.figure(figsize=(8, 5))
sns.countplot(x=y)
plt.title('Distribución de la variable objetivo (deposit)')
plt.xlabel('deposit')
plt.ylabel('Frecuencia')

# Añadir porcentajes
total = len(y)
for i, p in enumerate(plt.gca().patches):
    percentage = f'{100 * p.get_height() / total:.1f}%'
    plt.gca().annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()), 
                       ha='center', va='bottom')
plt.tight_layout()
plt.show()

# Determinar si está desbalanceado
proporciones = y.value_counts(normalize=True)
if proporciones.min() < 0.4:
    print("\n  El dataset está DESBALANCEADO")
    print(f"   Clase minoritaria: {proporciones.idxmin()} ({proporciones.min()*100:.2f}%)")
    print(f"   Clase mayoritaria: {proporciones.idxmax()} ({proporciones.max()*100:.2f}%)")
else:
    print("\n El dataset está balanceado")


# 1.7 RESUMEN DEL EDA


print("=" * 50)
print(" RESUMEN DEL EDA")
print("=" * 50)

resumen = {
    'Número de instancias': len(X),
    'Número de variables predictoras': X.shape[1],
    'Variables numéricas': len(num_cols),
    'Variables categóricas': len(cat_cols),
    'Variables con alta cardinalidad': len(alta_cardinalidad),
    'Variables con faltantes': len(missing_df),
    'Columnas constantes/ID': len(constantes) + len(posibles_ids),
    'Tipo de problema': 'Clasificación binaria',
    'Variable objetivo': target,
    'Clases': list(y.unique()),
    '¿Desbalanceado?': 'Sí' if proporciones.min() < 0.4 else 'No',
    'Tratamiento de pdays': 'Creada was_contacted_before + NaN'
}

for key, value in resumen.items():
    print(f"{key}: {value}")


# ---------------------------- 2. DECIDIR CÓMO SE VA A REALIZAR LA EVALUACIÓN ----------------------------


# 2.1 DEFINICIÓN DE LA MÉTRICA PRINCIPAL


print("=" * 50)
print("SELECCIÓN DE MÉTRICA")
print("=" * 50)

# Justificación: dado que hay desbalanceo, accuracy puede ser engañosa.
# F1-score balancea precisión y recall, mejor para clases desbalanceadas.

metrica = make_scorer(f1_score, average='binary', pos_label='yes')
print("Métrica elegida: F1-Score (binary, pos_label='yes')")
print("Justificación: Al haber desbalanceo, el F1-score es más robusto que la accuracy")


# 2.2 DIVISIÓN OUTER (TRAIN/TEST) - SOLO PARA EVALUACIÓN FINAL


print("=" * 50)
print("DIVISIÓN OUTER: TRAIN (2/3) - TEST (1/3)")
print("=" * 50)
print("   IMPORTANTE: Esta partición SOLO se usará al final para evaluación")
print("   Todo el desarrollo, HPO y comparaciones usarán SOLO train + CV\n")

# Usamos X_preprocesado que ya tiene el tratamiento de pdays
X_train_outer, X_test_outer, y_train_outer, y_test_outer = train_test_split(
    X_preprocesado, y, test_size=0.33, random_state=semilla, stratify=y
)

print(f"Tamaño del train set: {X_train_outer.shape}")
print(f"Tamaño del test set: {X_test_outer.shape}")
print(f"\nDistribución en train:")
print(y_train_outer.value_counts(normalize=True) * 100)
print(f"\nDistribución en test:")
print(y_test_outer.value_counts(normalize=True) * 100)


# 2.3 DEFINICIÓN DE LA EVALUACIÓN INNER


print("=" * 50)
print("CONFIGURACIÓN DE EVALUACIÓN INNER")
print("=" * 50)
print("La evaluación inner se usará para:")
print("  - Comparar diferentes modelos")
print("  - Optimización de hiperparámetros (HPO)")
print("  - Selección del mejor modelo\n")

# Configuración de validación cruzada para inner
cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
print(f"   Método inner: {cv_inner}")
print("   - 5 folds estratificados")
print("   - Shuffle=True para aleatorización")
print("   - Misma semilla para reproducibilidad")

# Función auxiliar para evaluar modelos con inner
def evaluar_modelo_inner(modelo, X, y, nombre_modelo, cv=cv_inner, scoring=metrica):
    """
    Evalúa un modelo usando validación cruzada sobre los datos de entrenamiento
    """
    start_time = time.time()
    scores = cross_val_score(modelo, X, y, cv=cv, scoring=scoring)
    elapsed_time = time.time() - start_time

    print(f"\n Resultados para {nombre_modelo}:")
    print(f"   F1-score medio: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")
    print(f"   Scores por fold: {[f'{s:.4f}' for s in scores]}")
    print(f"   Tiempo de entrenamiento (CV): {elapsed_time:.2f} segundos")
    
    return scores.mean(), scores.std(), elapsed_time


# 2.4 MODELO PARA COMPARACIÓN


print("=" * 50)
print("MODELO BASE (DUMMY) - LÍNEA BASE")
print("=" * 50)
print("Creamos un modelo trivial para comparar si nuestros modelos mejoran")
print("lo que sería una predicción ingenua (siempre predecir la clase mayoritaria)\n")

# Modelo dummy: siempre predice la clase más frecuente
modelo_dummy = DummyClassifier(strategy='most_frequent', random_state=semilla)

# Evaluar con inner
dummy_mean, dummy_std, dummy_time = evaluar_modelo_inner(
    modelo_dummy, X_train_outer, y_train_outer, "Dummy Classifier (most_frequent)"
)

print("\n  Este será nuestro punto de referencia:")
print(f"   Si los modelos no superan F1 ≈ {dummy_mean:.4f}, no son útiles")


# 2.5 PREPARACIÓN PARA PIPELINES


print("=" * 50)
print("PREPARACIÓN DE TRANSFORMACIONES PARA PIPELINES")
print("=" * 50)

# Identificar columnas por tipo (actualizado después del preproceso de pdays)
num_cols_final = X_train_outer.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_final = X_train_outer.select_dtypes(include=['object', 'category']).columns.tolist()

# Quitar 'was_contacted_before' de numéricas? No, es numérica binaria y está bien
print(f"Variables numéricas finales: {num_cols_final}")
print(f"Variables categóricas finales: {cat_cols_final}")

# Transformadores para pipelines (los usaremos en secciones siguientes)
# Numérico: imputación + escalado
transformador_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())  # Cambiaremos según el método
])

# Categórico: imputación + one-hot encoding
transformador_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Preprocesador completo
preprocesador = ColumnTransformer(transformers=[
    ('num', transformador_numerico, num_cols_final),
    ('cat', transformador_categorico, cat_cols_final)
])

print("\n Pipelines de preprocesamiento configurados:")
print(preprocesador)

# Guardamos el estado actual para poder retomar después
# (Opcional, pero útil)
joblib.dump(X_train_outer, 'X_train_outer.pkl')
joblib.dump(X_test_outer, 'X_test_outer.pkl')
joblib.dump(y_train_outer, 'y_train_outer.pkl')
joblib.dump(y_test_outer, 'y_test_outer.pkl')
joblib.dump(preprocesador, 'preprocesador.pkl')
print("\n Estado guardado para futuras sesiones")